In [ ]:
from elucidated_diffusion.elucidated_diffusion import edm_ancestral_sampling_for_diffusion
from elucidated_diffusion.elucidated_diffusion import edm_ancestral_sampling_for_sr
from elucidated_diffusion.elucidated_diffusion import P_mean, P_std, sigma_data, edm_loss_weight
from elucidated_diffusion.ema_helper import EMAHelper
from elucidated_diffusion.models.chatgpt_diffusion_unet import UNet128
from elucidated_diffusion.models.claude_skip_attention import SkipAttentionUNet
# from elucidated_diffusion.models.claude_cascaded_vit import FlexibleCascadedViT # old version
#from elucidated_diffusion.models.claude_cascaded_vit import MultiScaleSharedViT # old version in pixel space
from elucidated_diffusion.models.claude_cascaded_vit import SemanticCascadeViT

from elucidated_diffusion.models.claude_diffusion_ViTEnhancedUNet64 import create_vit_enhanced_unet
from elucidated_diffusion.models.claude_diffusion_ViTGradualTransition import create_gradual_transition_unet
from elucidated_diffusion.models.claude_diffusion_ViTGradualTransition import create_gradual_transition_unet
from elucidated_diffusion.models.claude_SemanticCoordinatorViTUNet import create_semantic_coordinator
from elucidated_diffusion.dataset_helpers import get_datasets
from elucidated_diffusion.checkpoint_helper import load_checkpoint, save_checkpoint, show_model_info

In [1]:

m = SemanticCascadeViT().to('cpu')
show_model_info(m)


Total parameters: 62,477,468
Trainable parameters: 62,477,468 (100.00%)

Parameter breakdown by top-level module:
model_256                           total:   20,315,699   trainable:   20,315,699   trainable%: 100.00
model_64                            total:   14,138,403   trainable:   14,138,403   trainable%: 100.00
model_32                            total:   14,015,523   trainable:   14,015,523   trainable%: 100.00
model_128                           total:   14,007,331   trainable:   14,007,331   trainable%: 100.00
default_semantic_token              total:          512   trainable:          512   trainable%: 100.00

Top 10 largest parameter tensors:
model_32.blocks.0.mlp.0.weight                               shape: (2048, 512) params:    1,048,576  train
model_32.blocks.0.mlp.2.weight                               shape: (512, 2048) params:    1,048,576  train
model_32.blocks.1.mlp.0.weight                               shape: (2048, 512) params:    1,048,576  train
model_32.blo

In [10]:
m = UNet128().to('cpu')
show_model_info(m)


Total parameters: 8,071,107
Trainable parameters: 8,071,107 (100.00%)

Parameter breakdown by top-level module:
up_block3                           total:    2,198,016   trainable:    2,198,016   trainable%: 100.00
down3                               total:    1,476,864   trainable:    1,476,864   trainable%: 100.00
mid                                 total:    1,476,864   trainable:    1,476,864   trainable%: 100.00
down2                               total:    1,214,976   trainable:    1,214,976   trainable%: 100.00
up_block2                           total:      655,872   trainable:      655,872   trainable%: 100.00
up3                                 total:      262,400   trainable:      262,400   trainable%: 100.00
down1                               total:      246,272   trainable:      246,272   trainable%: 100.00
up_block1                           total:      168,192   trainable:      168,192   trainable%: 100.00
up2                                 total:      131,200   traina

In [11]:

m = SkipAttentionUNet().to('cpu')
show_model_info(m)


Total parameters: 8,523,395
Trainable parameters: 8,523,395 (100.00%)

Parameter breakdown by top-level module:
up3                                 total:    2,460,416   trainable:    2,460,416   trainable%: 100.00
down3                               total:    1,476,864   trainable:    1,476,864   trainable%: 100.00
mid                                 total:    1,476,864   trainable:    1,476,864   trainable%: 100.00
down2                               total:    1,214,976   trainable:    1,214,976   trainable%: 100.00
up2                                 total:    1,082,112   trainable:    1,082,112   trainable%: 100.00
up1                                 total:      341,376   trainable:      341,376   trainable%: 100.00
down1                               total:      246,272   trainable:      246,272   trainable%: 100.00
up0                                 total:      160,576   trainable:      160,576   trainable%: 100.00
inc                                 total:       47,232   traina

In [12]:
m =  create_semantic_coordinator(config='balanced', img_size=128)
show_model_info(m)

Channel schedule: [64, 64, 256, 512]
Total parameters: 18,804,355
Trainable parameters: 18,804,355 (100.00%)

Parameter breakdown by top-level module:
cnn_decoder                         total:   11,744,707   trainable:   11,744,707   trainable%: 100.00
cnn_encoder                         total:    4,654,400   trainable:    4,654,400   trainable%: 100.00
vit_blocks                          total:    2,125,824   trainable:    2,125,824   trainable%: 100.00
from_vit                            total:      131,584   trainable:      131,584   trainable%: 100.00
to_vit                              total:      131,328   trainable:      131,328   trainable%: 100.00
time_mlp                            total:       16,512   trainable:       16,512   trainable%: 100.00

Top 10 largest parameter tensors:
cnn_decoder.blocks.0.conv1.weight                            shape: (512, 1024, 3, 3) params:    4,718,592  train
cnn_encoder.blocks.3.conv2.weight                            shape: (512, 512, 3, 

In [9]:
m = create_gradual_transition_unet("balanced").to('cpu')
show_model_info(m)


Total parameters: 10,407,283
Trainable parameters: 10,407,283 (100.00%)

Parameter breakdown by top-level module:
enc3                                total:    2,307,840   trainable:    2,307,840   trainable%: 100.00
dec4                                total:    1,490,240   trainable:    1,490,240   trainable%: 100.00
dec3                                total:    1,486,720   trainable:    1,486,720   trainable%: 100.00
enc4                                total:    1,285,120   trainable:    1,285,120   trainable%: 100.00
bottleneck1                         total:    1,285,120   trainable:    1,285,120   trainable%: 100.00
bottleneck2                         total:    1,285,120   trainable:    1,285,120   trainable%: 100.00
enc2                                total:      590,720   trainable:      590,720   trainable%: 100.00
dec2                                total:      378,560   trainable:      378,560   trainable%: 100.00
dec1                                total:      198,720   trai

In [13]:
m =  create_vit_enhanced_unet("balanced")
show_model_info(m)

Total parameters: 9,695,747
Trainable parameters: 9,695,747 (100.00%)

Parameter breakdown by top-level module:
vit_blocks                          total:    3,159,040   trainable:    3,159,040   trainable%: 100.00
up_block3                           total:    1,934,336   trainable:    1,934,336   trainable%: 100.00
down3                               total:    1,213,184   trainable:    1,213,184   trainable%: 100.00
down2                               total:      951,296   trainable:      951,296   trainable%: 100.00
up_block2                           total:      655,872   trainable:      655,872   trainable%: 100.00
mha_16                              total:      263,680   trainable:      263,680   trainable%: 100.00
mha_8                               total:      263,680   trainable:      263,680   trainable%: 100.00
up3                                 total:      262,400   trainable:      262,400   trainable%: 100.00
down1                               total:      246,272   traina

In [ ]:

#model_edm = SkipAttentionUNet().to(device) # Train one of these longer -- they're awesome at pokemon # 2025-12-02
#model_edm = MultiScaleSharedViT().to(device)
model_edm = SemanticCascadeViT().to(device)